# Summary — every model compared, and the website data

Run this **after** all model notebooks. It reads their saved forecasts (`results/predictions/`) and searches
(`results/metrics/`), then:
1. re-scores every model × case × split with one shared function, so every number is computed the same way;
2. shows the comparison tables (validation is used to compare; test is reported once);
3. exports the JSON files the website reads into `docs/data/`.

All website values are a **fraction of capacity** (power ÷ 21.5 MW), so no raw plant data is published.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Settings

In [ ]:
# Works whether the notebook is opened from src/ or from the project folder.
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
PRED = ROOT / "results" / "predictions"
METRICS = ROOT / "results" / "metrics"
WEB_DATA = ROOT / "docs" / "data"

CAPACITY_MW = 21.5
BAND = 0.07  # ±7% of capacity (GERC 2019 tolerance), as a fraction of capacity
DAY_FIRST_BLOCK, DAY_LAST_BLOCK = 29, 74  # 07:00-18:15

CASES = ["Case 1", "Case 2", "Case 3"]

# file name → model name, and how its forecast columns are named
LEARNING_MODELS = {
    "mlr": "MLR",
    "ridge": "Ridge",
    "lasso": "Lasso",
    "elastic_net": "Elastic Net",
    "random_forest": "RF",
    "adaboost": "AdaBoost",
}
DISPLAY = {"RF": "Random Forest"}

## 2. Collect every forecast into one long table

One row = one daytime block × one model × one case. Values are converted to a fraction of capacity here.

In [ ]:
def load(name):
    return pd.read_csv(PRED / f"{name}.csv", parse_dates=["TimeStamp", "Date"])


persistence = load("persistence")
base = persistence[["TimeStamp", "Date", "Block", "split", "actual"]].copy()
base["persistence"] = persistence["forecast"]

frames = []
for name, model in [("persistence", "Persistence"), ("seven_day_average", "7-day average")]:
    p = load(name)
    frames.append(p[["TimeStamp", "forecast"]].assign(model=model, case="—"))
for file, prefix in LEARNING_MODELS.items():
    p = load(file)
    for case in CASES:
        frames.append(p[["TimeStamp"]].assign(
            forecast=p[f"{prefix} {case}"], model=DISPLAY.get(prefix, prefix), case=case,
        ))

preds = pd.concat(frames, ignore_index=True).merge(base, on="TimeStamp")
preds = preds[preds["Block"].between(DAY_FIRST_BLOCK, DAY_LAST_BLOCK)].copy()
for col in ["actual", "forecast", "persistence"]:
    preds[col] = preds[col] / CAPACITY_MW

MODEL_ORDER = ["Persistence", "7-day average"] + [DISPLAY.get(m, m) for m in LEARNING_MODELS.values()]
print(preds.groupby(["model", "case"]).size().unstack().reindex(MODEL_ORDER))

## 3. One scoring function for everything

Inputs are fractions of capacity, so nMAE and nRMSE come out directly as % of capacity.
The skill score compares with persistence **on exactly the same blocks**.

In [ ]:
def score(part):
    err = part["forecast"] - part["actual"]
    err_p = part["persistence"] - part["actual"]
    rmse = np.sqrt((err**2).mean())
    return pd.Series({
        "nMAE (%)": 100 * err.abs().mean(),
        "nRMSE (%)": 100 * rmse,
        "Within ±7% (%)": 100 * (err.abs() <= BAND).mean(),
        "R² (%)": 100 * (1 - (err**2).sum() / ((part["actual"] - part["actual"].mean()) ** 2).sum()),
        "Skill vs persistence (%)": 100 * (1 - rmse / np.sqrt((err_p**2).mean())),
        "Blocks": len(part),
    })


table = preds.groupby(["model", "case", "split"]).apply(score, include_groups=False).reset_index()
table["model"] = pd.Categorical(table["model"], MODEL_ORDER, ordered=True)
table = table.sort_values(["model", "case", "split"]).reset_index(drop=True)
table.round(2)

## 4. Comparison tables

**Validation** (used to compare models — skill vs persistence, %):

In [ ]:
def pivot(split, metric):
    t = table[table["split"] == split].pivot(index="model", columns="case", values=metric)
    return t.reindex(MODEL_ORDER).round(2)


pivot("val", "Skill vs persistence (%)")

In [ ]:
pivot("val", "nMAE (%)")

In [ ]:
pivot("val", "Within ±7% (%)")

**Test** (reported once, never used for choices — skill vs persistence, %):

In [ ]:
pivot("test", "Skill vs persistence (%)")

Best model per case on validation (lowest nRMSE), and how it then did on test:

In [ ]:
learning = table[table["case"] != "—"]
val = learning[learning["split"] == "val"]
winners = val.loc[val.groupby("case")["nRMSE (%)"].idxmin(), ["case", "model", "nRMSE (%)", "Skill vs persistence (%)"]]
test = learning[learning["split"] == "test"].set_index(["case", "model"])["Skill vs persistence (%)"]
winners["test skill (%)"] = [test.loc[(c, m)] for c, m in zip(winners["case"], winners["model"])]
winners.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, case in zip(axes, CASES):
    part = learning[learning["case"] == case].pivot(index="model", columns="split", values="Skill vs persistence (%)")
    part = part.reindex([m for m in MODEL_ORDER if m in part.index])[["val", "test"]]
    part.plot.bar(ax=ax, rot=45)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(case)
    ax.set_xlabel("")
axes[0].set_ylabel("Skill vs persistence (%)")
plt.tight_layout()
plt.show()

## 5. Per-day scores (every day: train, validation and test)

Used by the website's day readout and the "where does each model fail" view.

In [ ]:
holdout = preds  # every scored day: train, validation and test
daily = holdout.groupby(["Date", "split", "model", "case"]).apply(score, include_groups=False).reset_index()
daily = daily.drop(columns="Blocks")

# Hardest days for the realistic case (mean nMAE over all Case 2 models)
daily[daily["case"] == "Case 2"].groupby("Date")["nMAE (%)"].mean().sort_values(ascending=False).head(5).round(2)

## 5b. How sure are we? Bootstrap over days

The test set has 7 days and validation 24, so a skill score could move a lot with a few different days.
Resample the days with replacement 1,000 times, recompute each skill score, and keep the middle 90%.
If an interval crosses 0, the data cannot tell that model apart from persistence.

In [ ]:
rng = np.random.default_rng(0)
N_BOOT = 1000

per_day = (
    preds.assign(se=(preds["forecast"] - preds["actual"]) ** 2, se_p=(preds["persistence"] - preds["actual"]) ** 2)
    .groupby(["model", "case", "split", "Date"], observed=True)[["se", "se_p"]].sum()
    .reset_index()
)

ci = []
for (model, case, split), g in per_day.groupby(["model", "case", "split"], observed=True):
    se, se_p = g["se"].to_numpy(), g["se_p"].to_numpy()
    idx = rng.integers(0, len(g), size=(N_BOOT, len(g)))
    skill = 100 * (1 - np.sqrt(se[idx].sum(1) / se_p[idx].sum(1)))
    ci.append({"model": model, "case": case, "split": split,
               "skill 5%": np.percentile(skill, 5), "skill 95%": np.percentile(skill, 95)})
ci = pd.DataFrame(ci)
table = table.drop(columns=["skill 5%", "skill 95%"], errors="ignore").merge(ci, on=["model", "case", "split"])

t = table[(table["split"] == "test") & (table["case"] != "—")]
t[["model", "case", "Skill vs persistence (%)", "skill 5%", "skill 95%"]].round(1)

## 6. Export the website data (`docs/data/`)

| File | Used by |
|---|---|
| `results.json` | home page results table, MAE vs RMSE ranking toggle |
| `days.json` | day explorer (curves + per-day scores for every day) |
| `regularisation.json` | Ridge / Lasso / Elastic Net penalty slider |
| `random_forest.json` | depth slider and input importance |
| `adaboost.json` | rounds slider (errors, average loss, vote weight, forecasts after selected rounds) |

In [ ]:
WEB_DATA.mkdir(parents=True, exist_ok=True)


def r4(x):
    # round floats for small files; keep None for missing values
    if isinstance(x, (list, tuple, np.ndarray, pd.Series)):
        return [r4(v) for v in x]
    return None if pd.isna(x) else round(float(x), 4)


def save(name, obj):
    path = WEB_DATA / name
    path.write_text(json.dumps(obj, separators=(",", ":")))
    print(f"{name}: {path.stat().st_size / 1024:.0f} KB")

**results.json** — every model × case × split:

In [ ]:
save("results.json", {
    "note": "All errors are % of the plant's 21.5 MW capacity, daytime blocks 07:00-18:15 only.",
    "models": MODEL_ORDER,
    "cases": CASES,
    "rows": [
        {"model": str(r["model"]), "case": r["case"], "split": r["split"],
         "nMAE": r4(r["nMAE (%)"]), "nRMSE": r4(r["nRMSE (%)"]), "within_band": r4(r["Within ±7% (%)"]),
         "r2": r4(r["R² (%)"]), "skill": r4(r["Skill vs persistence (%)"]), "blocks": int(r["Blocks"]),
         "skill_lo": r4(r["skill 5%"]), "skill_hi": r4(r["skill 95%"])}
        for r in table.to_dict("records")
    ],
})

**days.json** — actual and every forecast for each day (train, validation, test), plus that day's scores:

In [ ]:
blocks = list(range(DAY_FIRST_BLOCK, DAY_LAST_BLOCK + 1))
days = []
for (date, split), d in holdout.groupby(["Date", "split"]):
    actual = d[d["model"] == "Persistence"].sort_values("Block")["actual"]
    forecasts = {
        f"{m}|{c}": r4(g.sort_values("Block")["forecast"])
        for (m, c), g in d.groupby(["model", "case"], observed=True)
    }
    day_scores = daily[daily["Date"] == date]
    days.append({
        "date": date.strftime("%Y-%m-%d"),
        "split": split,
        "actual": r4(actual),
        "forecasts": forecasts,
        "scores": {
            f"{r['model']}|{r['case']}": {
                "nMAE": r4(r["nMAE (%)"]), "nRMSE": r4(r["nRMSE (%)"]),
                "within_band": r4(r["Within ±7% (%)"]), "skill": r4(r["Skill vs persistence (%)"]),
            }
            for r in day_scores.to_dict("records")
        },
    })

save("days.json", {"blocks": blocks, "hours": [(b - 1) / 4 for b in blocks], "days": days})

**regularisation.json** — for every α: training/validation nRMSE and the standardised coefficients
(as a fraction of capacity per 1 standard deviation of the input), plus the α chosen on validation.

In [ ]:
def regularisation_block(file, extra_key=None):
    s = pd.read_csv(METRICS / file)
    out = {}
    for case, part in s.groupby("case"):
        coef_cols = [c for c in part.columns if c.startswith("coef ") and part[c].notna().any()]
        groups = part.groupby(extra_key) if extra_key else [(None, part)]
        series = {}
        for key, g in groups:
            g = g.sort_values("alpha")
            series[str(key) if key is not None else "all"] = {
                "alpha": r4(g["alpha"]),
                "train_nRMSE": r4(100 * g["train RMSE (MW)"] / CAPACITY_MW),
                "val_nRMSE": r4(100 * g["val RMSE (MW)"] / CAPACITY_MW),
                "coefs": {c.removeprefix("coef "): r4(g[c] / CAPACITY_MW) for c in coef_cols},
            }
        chosen = part.loc[part["val RMSE (MW)"].idxmin()]
        out[case] = {
            "series": series,
            "chosen_alpha": r4(chosen["alpha"]),
            **({"chosen_l1_ratio": r4(chosen[extra_key])} if extra_key else {}),
        }
    return out


save("regularisation.json", {
    "Ridge": regularisation_block("ridge_alpha_search.csv"),
    "Lasso": regularisation_block("lasso_search.csv"),
    "Elastic Net": regularisation_block("elastic_net_search.csv", extra_key="l1_ratio"),
})

**random_forest.json** — depth sweep (training vs validation nRMSE), chosen settings, input importance:

In [ ]:
rf_search = pd.read_csv(METRICS / "random_forest_search.csv")
rf_sweep = pd.read_csv(METRICS / "random_forest_depth_sweep.csv")
rf_imp = pd.read_csv(METRICS / "random_forest_importance.csv", index_col=0)

rf = {}
for case in CASES:
    chosen = rf_search[rf_search["case"] == case].sort_values("val RMSE (MW)").iloc[0]
    sw = rf_sweep[rf_sweep["case"] == case]
    rf[case] = {
        "chosen": {
            "max_depth": None if pd.isna(chosen["max_depth"]) else int(chosen["max_depth"]),
            "min_samples_leaf": int(chosen["min_samples_leaf"]),
            "max_features": r4(chosen["max_features"]),
        },
        "depth": [None if pd.isna(d) else int(d) for d in sw["max_depth"]],
        "train_nRMSE": r4(100 * sw["train RMSE (MW)"] / CAPACITY_MW),
        "val_nRMSE": r4(100 * sw["val RMSE (MW)"] / CAPACITY_MW),
        "importance": {k: r4(100 * v / CAPACITY_MW) for k, v in rf_imp[case].dropna().items()},
    }
save("random_forest.json", rf)

**adaboost.json** — per-round errors, average loss and vote weight; forecasts after selected rounds:

In [ ]:
ada_search = pd.read_csv(METRICS / "adaboost_search.csv")
ada_rounds = pd.read_csv(METRICS / "adaboost_rounds.csv")
ada_imp = pd.read_csv(METRICS / "adaboost_importance.csv", index_col=0)
ada_staged = pd.read_csv(PRED / "adaboost_by_round.csv", parse_dates=["TimeStamp"])
ada_staged = ada_staged.assign(Date=ada_staged["TimeStamp"].dt.strftime("%Y-%m-%d"), forecast=ada_staged["forecast"] / CAPACITY_MW)

ada = {}
for case in CASES:
    chosen = ada_search[ada_search["case"] == case].sort_values("val RMSE (MW)").iloc[0]
    rd = ada_rounds[ada_rounds["case"] == case]
    st = ada_staged[ada_staged["case"] == case].sort_values("TimeStamp")
    ada[case] = {
        "chosen": {
            "base_depth": int(chosen["base_depth"]),
            "learning_rate": r4(chosen["learning_rate"]),
            "loss": chosen["loss"],
            "rounds": int(chosen["best rounds"]),
        },
        "round": rd["round"].tolist(),
        "train_nRMSE": r4(100 * rd["train RMSE (MW)"] / CAPACITY_MW),
        "val_nRMSE": r4(100 * rd["val RMSE (MW)"] / CAPACITY_MW),
        "average_loss": r4(rd["average loss"]),
        "vote_weight": r4(rd["vote weight"]),
        "importance": {k: r4(100 * v / CAPACITY_MW) for k, v in ada_imp[case].dropna().items()},
        "forecast_by_round": {
            str(t): {date: r4(g["forecast"]) for date, g in part.groupby("Date")}
            for t, part in st.groupby("round")
        },
    }
save("adaboost.json", ada)

**data.json** — data overview for the home page: daily energy across the season, the average daily profile
per split, a sample of power vs tomorrow's GHI (training, daytime), and correlations between the inputs.

In [ ]:
clean = pd.read_csv(ROOT / "Data" / "Clean" / "Winter_Clean.csv", parse_dates=["TimeStamp", "Date"]).rename(columns={
    "Act. Power (MW)": "power", "W-GHI (W/m2)": "ghi", "W-MODT- (°C)": "modt",
    "W-AMBT (°C)": "ambt", "W-WSPD (m/s)": "wspd", "W-WDIR- (Degree)": "wdir",
})
WEATHER = ["ghi", "modt", "ambt", "wspd", "wdir"]
clean["frac"] = clean["power"] / CAPACITY_MW
split_of = base.drop_duplicates("Date").set_index("Date")["split"]
clean["split"] = clean["Date"].map(split_of).fillna("train")  # 15 Oct is only an input day, in training

by_block = clean.groupby("Block")
for col in ["frac"] + WEATHER:
    clean[f"{col}_D"] = by_block[col].shift(1)
for w in WEATHER:
    clean[f"{w}_D+1"] = clean[w]
lagged = clean.dropna(subset=["frac_D"])
train_day = lagged[(lagged["split"] == "train") & lagged["Block"].between(DAY_FIRST_BLOCK, DAY_LAST_BLOCK)]

energy = clean.groupby("Date").agg(energy=("frac", lambda s: s.sum() / 4), split=("split", "first")).reset_index()
profile = clean.groupby(["split", "Block"])["frac"].mean().unstack(0)
sample = train_day.sample(1500, random_state=0)

CORR_COLS = ["frac", "frac_D", "ghi_D", "modt_D", "ambt_D", "wspd_D", "wdir_D", "ghi_D+1", "modt_D+1", "ambt_D+1"]
corr = train_day[CORR_COLS].corr()
labels = [c.replace("frac_D", "power_D").replace("frac", "power (target)") for c in CORR_COLS]

save("data.json", {
    "energy": {"date": energy["Date"].dt.strftime("%Y-%m-%d").tolist(), "split": energy["split"].tolist(),
               "capacity_hours": r4(energy["energy"])},
    "profile": {"hour": [(b - 1) / 4 for b in profile.index], **{s: r4(profile[s]) for s in ["train", "val", "test"]}},
    "scatter": {"ghi_next": r4(sample["ghi_D+1"]), "power": r4(sample["frac"]), "hour": r4((sample["Block"] - 1) / 4)},
    "corr": {"labels": labels, "z": [r4(row) for row in corr.to_numpy()]},
})

**models.json** — inputs for the model pages' step-throughs:
- `mlr_example`: the six real rows (block 46, 24–29 Oct) used to derive least squares by hand, as % of capacity;
- `mlr_coefficients`: the fitted MLR coefficients per case;
- `tree_rows`: every training daytime row with the Case 2 and Case 3 inputs, so the page can recompute a regression tree's splits;
- `boost_walk`: 20 training blocks (GHI → power) with scikit-learn's own AdaBoost.R2 stumps, losses, β, votes and weights per round.

In [ ]:
blk = lagged[lagged["Block"] == 46].set_index("Date")
example = blk.loc["2025-10-24":"2025-10-29"]

# AdaBoost.R2 walkthrough: 20 training daytime blocks (x = GHI at that block, y = power), stumps, learning rate 1.
# The first sample (seed 0, 1, ...) on which boosting runs at least 6 rounds is used, so there is something to step through.
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor

for seed in range(50):
    pick = train_day.sample(20, random_state=seed).sort_values("ghi_D+1")
    X_b, y_b = pick[["ghi_D+1"]].to_numpy(), 100 * pick["frac"].to_numpy()
    ada_walk = AdaBoostRegressor(DecisionTreeRegressor(max_depth=1), n_estimators=8, learning_rate=1.0,
                                 loss="linear", random_state=0).fit(X_b, y_b)
    if (ada_walk.estimator_weights_ > 0).sum() >= 6:
        break
print("walkthrough sample seed:", seed)

w = np.full(len(y_b), 1 / len(y_b))
walk_rounds = []
for est, lbar_sk in zip(ada_walk.estimators_, ada_walk.estimator_errors_):
    pred = est.predict(X_b)
    err = np.abs(y_b - pred)
    loss = err / err.max()
    lbar = float((w * loss).sum())
    assert abs(lbar - lbar_sk) < 1e-9  # same average loss as scikit-learn
    beta = lbar / (1 - lbar)
    tree = est.tree_
    walk_rounds.append({
        "threshold": r4(tree.threshold[0]), "left": r4(tree.value[1].ravel()[0]), "right": r4(tree.value[2].ravel()[0]),
        "pred": r4(pred), "loss": r4(loss), "lbar": r4(lbar), "beta": r4(beta), "vote": r4(np.log(1 / beta)),
        "w_before": r4(w),
    })
    w = w * beta ** (1 - loss)
    w = w / w.sum()
    walk_rounds[-1]["w_after"] = r4(w)

grid = np.linspace(0, 950, 191)
walk_staged = [r4(p) for p in ada_walk.staged_predict(grid.reshape(-1, 1))]

mlr_coef = pd.read_csv(METRICS / "mlr_coefficients.csv", index_col=0)
tree_inputs = ["frac_D", "ghi_D", "modt_D", "ambt_D", "wspd_D", "wdir_D", "ghi_D+1", "modt_D+1", "ambt_D+1", "wspd_D+1", "wdir_D+1"]

save("models.json", {
    "mlr_example": {"date": example.index.strftime("%Y-%m-%d").tolist(),
                    "x": r4(100 * example["frac_D"]), "y": r4(100 * example["frac"])},
    "mlr_coefficients": {c: {k: r4(v) for k, v in mlr_coef[c].dropna().items()} for c in CASES},
    "tree_rows": {"inputs": [c.replace("frac_D", "power_D") for c in tree_inputs],
                  "X": [r4(train_day[c]) for c in tree_inputs], "y": r4(train_day["frac"])},
    "boost_walk": {"date": pick["Date"].dt.strftime("%Y-%m-%d").tolist(), "hour": r4((pick["Block"] - 1) / 4),
                   "x": r4(X_b.ravel()), "y": r4(y_b), "rounds": walk_rounds,
                   "grid": r4(grid), "staged": walk_staged},
})

## 6b. Two experiments for the website (`experiments.json`)

Both refit the six models with the settings already chosen on validation, so nothing here is used to tune anything.

1. **How good must the weather forecast be?** Case 3 uses tomorrow's *measured* weather. Here tomorrow's sunlight is
   given a forecast error: each day's sunlight is multiplied by (1 + ε), with ε drawn from a normal distribution with
   standard deviation σ (one draw per day, because a forecast is usually wrong for the whole day). Module temperature is
   shifted consistently (its excess over air temperature scales with sunlight). Skill is averaged over 30 random draws.
2. **Break the evaluation.** The same models are scored with the days shuffled, and with individual 15-minute blocks
   shuffled, instead of the chronological split. Shuffling lets a model train on blocks from the same day (or the day
   after) as the ones it is tested on — information it would never have in real use.

In [ ]:
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

# Daytime rows of every split, in MW, with the same input names as the model notebooks.
rows_all = lagged[lagged["Block"].between(DAY_FIRST_BLOCK, DAY_LAST_BLOCK)].copy()
rows_all["power_D"] = clean.groupby("Block")["power"].shift(1).loc[rows_all.index]  # exact MW values, as in the notebooks
FEATS = {
    "Case 1": ["power_D"],
    "Case 2": ["power_D"] + [f"{w}_D" for w in WEATHER],
    "Case 3": ["power_D"] + [f"{w}_D" for w in WEATHER] + [f"{w}_D+1" for w in WEATHER],
}


def chosen(file):
    s = pd.read_csv(METRICS / file)
    return s.loc[s.groupby("case")["val RMSE (MW)"].idxmin()].set_index("case")


ridge_c, lasso_c, enet_c = chosen("ridge_alpha_search.csv"), chosen("lasso_search.csv"), chosen("elastic_net_search.csv")
rf_c, ada_c = chosen("random_forest_search.csv"), chosen("adaboost_search.csv")


def make_model(name, case):
    if name == "MLR":
        return LinearRegression()
    if name == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=ridge_c.loc[case, "alpha"]))
    if name == "Lasso":
        return make_pipeline(StandardScaler(), Lasso(alpha=lasso_c.loc[case, "alpha"], max_iter=50_000))
    if name == "Elastic Net":
        return make_pipeline(StandardScaler(), ElasticNet(alpha=enet_c.loc[case, "alpha"], l1_ratio=enet_c.loc[case, "l1_ratio"], max_iter=50_000))
    if name == "Random Forest":
        b = rf_c.loc[case]
        return RandomForestRegressor(n_estimators=300, max_depth=None if pd.isna(b["max_depth"]) else int(b["max_depth"]),
                                     min_samples_leaf=int(b["min_samples_leaf"]), max_features=b["max_features"], random_state=42, n_jobs=-1)
    b = ada_c.loc[case]
    return AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=int(b["base_depth"])), n_estimators=int(b["best rounds"]),
                             learning_rate=b["learning_rate"], loss=b["loss"], random_state=42)


def skill_of(model, part, features, X=None):
    pred = np.clip(model.predict(part[features] if X is None else X), 0, CAPACITY_MW)
    rmse = np.sqrt(((pred - part["power"]) ** 2).mean())
    rmse_p = np.sqrt(((part["power_D"] - part["power"]) ** 2).mean())
    return 100 * (1 - rmse / rmse_p), 100 * rmse / CAPACITY_MW


MODEL_NAMES = ["MLR", "Ridge", "Lasso", "Elastic Net", "Random Forest", "AdaBoost"]
tr_rows = rows_all[rows_all["split"] == "train"]

**Experiment 1 — forecast error.** Fit each Case 3 model once on training, then degrade tomorrow's sunlight at prediction time.

In [ ]:
LEVELS = [0, 5, 10, 15, 20, 25, 30, 40, 50]  # σ of the day-level sunlight error, in %
N_DRAWS = 30
f3 = FEATS["Case 3"]
case3 = {m: make_model(m, "Case 3").fit(tr_rows[f3], tr_rows["power"]) for m in MODEL_NAMES}

forecast_error = {m: {"val": [], "test": []} for m in MODEL_NAMES}
for split in ["val", "test"]:
    part = rows_all[rows_all["split"] == split]
    day_code = part["Date"].factorize()[0]
    for sigma in LEVELS:
        totals = {m: 0.0 for m in MODEL_NAMES}
        for draw in range(N_DRAWS):
            eps = np.random.default_rng(1000 + draw).normal(0, sigma / 100, day_code.max() + 1)[day_code]
            X = part[f3].copy()
            X["ghi_D+1"] = np.clip(part["ghi_D+1"] * (1 + eps), 0, None)
            X["modt_D+1"] = part["ambt_D+1"] + (part["modt_D+1"] - part["ambt_D+1"]) * (1 + eps)
            for m in MODEL_NAMES:
                totals[m] += skill_of(case3[m], part, f3, X)[0]
        for m in MODEL_NAMES:
            forecast_error[m][split].append(totals[m] / N_DRAWS)

# With no error this must reproduce the main results.
for m in MODEL_NAMES:
    ref = table[(table["model"] == m) & (table["case"] == "Case 3") & (table["split"] == "test")]["Skill vs persistence (%)"].iloc[0]
    assert abs(forecast_error[m]["test"][0] - ref) < 0.05, (m, forecast_error[m]["test"][0], ref)

pd.DataFrame({m: forecast_error[m]["test"] for m in MODEL_NAMES}, index=pd.Index(LEVELS, name="sunlight error σ (%)")).round(1)

**Experiment 2 — shuffled splits.** Same models and settings; only the way rows are divided changes. Each shuffled
scheme is repeated with 5 seeds and averaged. Sizes match the real split (91 / 24 / 7 days, or the same share of blocks).

In [ ]:
dates = np.array(sorted(rows_all["Date"].unique()))
n_train_days, n_test_days = 91, 7


def shuffled_parts(scheme, seed):
    rng = np.random.default_rng(seed)
    if scheme == "shuffled days":
        d = rng.permutation(dates)
        return rows_all[rows_all["Date"].isin(d[:n_train_days])], rows_all[rows_all["Date"].isin(d[-n_test_days:])]
    idx = rng.permutation(len(rows_all))
    n_tr, n_te = len(tr_rows), int((rows_all["split"] == "test").sum())
    return rows_all.iloc[idx[:n_tr]], rows_all.iloc[idx[-n_te:]]


leakage = []
for case, feats in FEATS.items():
    for m in MODEL_NAMES:
        fitted = make_model(m, case).fit(tr_rows[feats], tr_rows["power"])
        s, r = skill_of(fitted, rows_all[rows_all["split"] == "test"], feats)
        leakage.append({"model": m, "case": case, "scheme": "chronological", "skill": s, "nRMSE": r})
        for scheme in ["shuffled days", "shuffled blocks"]:
            out = []
            for seed in range(5):
                tr_s, te_s = shuffled_parts(scheme, seed)
                out.append(skill_of(make_model(m, case).fit(tr_s[feats], tr_s["power"]), te_s, feats))
            leakage.append({"model": m, "case": case, "scheme": scheme, "skill": np.mean([o[0] for o in out]), "nRMSE": np.mean([o[1] for o in out])})
leakage = pd.DataFrame(leakage)

# The chronological rows must reproduce the main results exactly.
for r in leakage[leakage["scheme"] == "chronological"].itertuples():
    ref = table[(table["model"] == r.model) & (table["case"] == r.case) & (table["split"] == "test")]["Skill vs persistence (%)"].iloc[0]
    assert abs(r.skill - ref) < 0.05, (r.model, r.case, r.skill, ref)

leakage[leakage["case"] == "Case 2"].pivot(index="model", columns="scheme", values="skill").reindex(MODEL_NAMES).round(1)

In [ ]:
save("experiments.json", {
    "forecast_error": {"levels": LEVELS, "draws": N_DRAWS,
                       "skill": {m: {s: r4(v) for s, v in d.items()} for m, d in forecast_error.items()}},
    "leakage": [{"model": r["model"], "case": r["case"], "scheme": r["scheme"], "skill": r4(r["skill"]), "nRMSE": r4(r["nRMSE"])}
                for r in leakage.to_dict("records")],
})

## 6c. How the models learn (`learning.json`)

Four small studies for the website. Like section 6b, they reuse the settings already chosen on validation.

1. **Error surface for gradient descent.** For Case 1 the squared error of the line ŷ = b₀ + b₁·x depends on the data only
   through five averages (mean of x, y, x², xy, y²), so exporting those reproduces the surface exactly.
2. **Inputs kept or dropped.** Standardised coefficients of the four linear models and permutation importance of the two
   tree models, per case.
3. **Bias and variance.** Refit on 50 bootstrap resamples of the training *days* and predict the validation blocks each time.
   At each block, bias² = (average forecast − actual)² and variance = spread of the forecasts across resamples; their sum is the
   expected squared error. (Bias² here also contains the noise that no model could predict, since each block is observed once.)
   A forest of n trees is the mean of the first n trees of a 100-tree forest.
4. **Learning curve.** Train on only the most recent N training days and score on validation.

In [ ]:
val_rows = rows_all[rows_all["split"] == "val"]
train_dates = np.array(sorted(tr_rows["Date"].unique()))
to_pct = 100 / CAPACITY_MW

# 1. Moments of the Case 1 problem, in % of capacity
gx, gy = tr_rows["power_D"] * to_pct, tr_rows["power"] * to_pct
moments = {"mx": gx.mean(), "my": gy.mean(), "mxx": (gx * gx).mean(), "mxy": (gx * gy).mean(), "myy": (gy * gy).mean(), "n": len(gx)}
slope = (moments["mxy"] - moments["mx"] * moments["my"]) / (moments["mxx"] - moments["mx"] ** 2)
print(f"least-squares line from the moments: {moments['my'] - slope * moments['mx']:.3f} + {slope:.4f}·x  (in % of capacity)")

# 2. Inputs kept or dropped
features = {}
for case, feats in FEATS.items():
    coef = {}
    for m in ["MLR", "Ridge", "Lasso", "Elastic Net"]:
        pipe = make_pipeline(StandardScaler(), LinearRegression()) if m == "MLR" else make_model(m, case)
        coef[m] = r4(pipe.fit(tr_rows[feats], tr_rows["power"])[-1].coef_ * to_pct + 0.0)
    features[case] = {
        "inputs": feats, "coef": coef,
        "importance": {"Random Forest": r4(rf_imp[case].reindex(feats) * to_pct), "AdaBoost": r4(ada_imp[case].reindex(feats) * to_pct)},
    }
{case: {m: [f for f, c in zip(v["inputs"], v["coef"][m]) if c == 0] for m in ["Lasso", "Elastic Net"]} for case, v in features.items()}

In [ ]:
# 3. Bias and variance on validation, from 50 resamples of the training days
N_RESAMPLES = 50
TREE_COUNTS = [1, 2, 3, 5, 10, 20, 50, 100]
by_date = {d: g for d, g in tr_rows.groupby("Date")}
y_val = val_rows["power"].to_numpy() * to_pct


def decompose(preds):  # preds: resamples × validation blocks, in % of capacity
    preds = np.asarray(preds)
    return {"bias2": float(((preds.mean(0) - y_val) ** 2).mean()), "variance": float(preds.var(0).mean())}


bias_variance = {}
for case, feats in FEATS.items():
    b = rf_c.loc[case]
    X_val = val_rows[feats].to_numpy()
    mlr_preds, forest_preds = [], {n: [] for n in TREE_COUNTS}
    for i in range(N_RESAMPLES):
        picked = np.random.default_rng(500 + i).choice(train_dates, len(train_dates), replace=True)
        sample = pd.concat([by_date[d] for d in picked])
        mlr_preds.append(LinearRegression().fit(sample[feats].to_numpy(), sample["power"]).predict(X_val) * to_pct)
        forest = RandomForestRegressor(
            n_estimators=max(TREE_COUNTS), max_depth=None if pd.isna(b["max_depth"]) else int(b["max_depth"]),
            min_samples_leaf=int(b["min_samples_leaf"]), max_features=b["max_features"], random_state=i, n_jobs=-1,
        ).fit(sample[feats].to_numpy(), sample["power"])
        running = np.cumsum([t.predict(X_val) for t in forest.estimators_], axis=0) * to_pct
        for n in TREE_COUNTS:
            forest_preds[n].append(running[n - 1] / n)
    bias_variance[case] = {"mlr": decompose(mlr_preds), "forest": [decompose(forest_preds[n]) for n in TREE_COUNTS]}

pd.DataFrame({f"{case} {k}": [d[k] for d in v["forest"]] + [v["mlr"][k]] for case, v in bias_variance.items() for k in ["bias2", "variance"]},
             index=[f"forest, {n} trees" for n in TREE_COUNTS] + ["MLR"]).round(1)

In [ ]:
# 4. Learning curve: the most recent N training days
DAY_COUNTS = [7, 14, 21, 30, 45, 60, 75, 91]
nrmse = lambda model, part, feats: float(100 * np.sqrt(((np.clip(model.predict(part[feats]), 0, CAPACITY_MW) - part["power"]) ** 2).mean()) / CAPACITY_MW)

learning_curve = {}
for case, feats in FEATS.items():
    learning_curve[case] = {"persistence_val": float(100 * np.sqrt(((val_rows["power_D"] - val_rows["power"]) ** 2).mean()) / CAPACITY_MW)}
    for m in ["MLR", "Random Forest"]:
        tr_err, val_err = [], []
        for n in DAY_COUNTS:
            recent = tr_rows[tr_rows["Date"] >= train_dates[-n]]
            fitted = make_model(m, case).fit(recent[feats], recent["power"])
            tr_err.append(nrmse(fitted, recent, feats))
            val_err.append(nrmse(fitted, val_rows, feats))
        learning_curve[case][m] = {"train": tr_err, "val": val_err}

# With all 91 days this must reproduce the main validation results.
for case in FEATS:
    for m, name in [("MLR", "MLR"), ("Random Forest", "Random Forest")]:
        ref = table[(table["model"] == name) & (table["case"] == case) & (table["split"] == "val")]["nRMSE (%)"].iloc[0]
        assert abs(learning_curve[case][m]["val"][-1] - ref) < 0.01, (case, m, learning_curve[case][m]["val"][-1], ref)

pd.DataFrame({f"{case} {m}": learning_curve[case][m]["val"] for case in FEATS for m in ["MLR", "Random Forest"]},
             index=pd.Index(DAY_COUNTS, name="training days")).round(2)

In [ ]:
save("learning.json", {
    "moments": {k: (int(v) if k == "n" else round(float(v), 6)) for k, v in moments.items()},
    "features": features,
    "bias_variance": {"tree_counts": TREE_COUNTS, "resamples": N_RESAMPLES,
                      "cases": {case: {"mlr": {k: r4(x) for k, x in v["mlr"].items()},
                                       "forest": [{k: r4(x) for k, x in d.items()} for d in v["forest"]]} for case, v in bias_variance.items()}},
    "learning_curve": {"days": DAY_COUNTS,
                       "cases": {case: {"persistence_val": r4(v["persistence_val"]),
                                        **{m: {s: r4(e) for s, e in v[m].items()} for m in ["MLR", "Random Forest"]}} for case, v in learning_curve.items()}},
})

## 7. Self-check

The website must show exactly the notebook numbers. Re-read `results.json` and compare it with the table above.

In [ ]:
check = pd.DataFrame(json.loads((WEB_DATA / "results.json").read_text())["rows"])
merged = check.merge(table.assign(model=table["model"].astype(str)), on=["model", "case", "split"])
diff = (merged["nRMSE"] - merged["nRMSE (%)"]).abs().max()
print(f"rows: {len(check)}, largest nRMSE difference: {diff:.5f} (rounding only)")
assert diff < 1e-3